In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
np.float_ = np.float64
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid', 'c2_kegg_medicus']
clonemodes = ['scatrex', 'phenograph']
gene_set_collection = gene_set_collections[3]
clonemode = 'scatrex'
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
    
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
concentration_DMSO = '100'
concentration_drug = '5'
path_info_cohort = None
model_scatrex = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref = model_scatrex.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


clonemode = 'phenograph'
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
model_phenograph = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref_phenograph = model_phenograph.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

# Comparing scatrex vs phenograph 

In [ ]:
def get_stat(model, COHORT, gene_set_collection, clonemode_stat, pl1=3.2, pl2=3.2, MODE='TEST'):
#         ite = 0
#         found = False
#         while (not(found) and ite<10):

#             try:
#                 with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode_stat, ite)), 'rb') as handle:
#                     params = pickle.load(handle)
#                     dim = params['beta'].shape[1]
#                 found = True
#             except:
#                 ite+=1
#                 pass
    dim = len(model.feature_names)
    param_studied = 'beta' # ['beta', 'pi']        

    if param_studied == 'beta':
        beta_mean = np.zeros((47,dim))
        beta2_mean = np.zeros((47,dim))
    else:
        D = params['beta'].shape[0]
        N, Kmax = params['proportions'].shape
        beta_mean = np.zeros((D, Kmax, 30))
        beta2_mean = np.zeros((D, Kmax, 30))
    
    count = 0
    all_data = [{},{}]
    corrs = {}
    nbites= 95
    for ite in range(nbites):
        try:
            with open(os.path.join(rootpath,'{0}/{1}_{2}/stability_paper/params_svi_ite_{3}_l1_{4}_l2_{5}.pkl'.format(COHORT, gene_set_collection, clonemode_stat, ite, pl1, pl2)), 'rb') as handle:
                params = pickle.load(handle)
                if param_studied == 'beta':
                    beta_mean += params['beta'] /nbites
                    beta2_mean += params['beta']**2 /nbites
                    
                    samples_names_train = params['sample_names_train']
                    samples_names_test = params['sample_names_test']

                    idxs_train = []
                    idxs_test = []
                    for sample in samples_names_train:
                        idx_s = np.where(model.sample_names==sample)[0][0]
                        idxs_train.append(idx_s)
                    for sample in samples_names_test:
                        idx_s = np.where(model.sample_names==sample)[0][0]
                        idxs_test.append(idx_s)
                    
                    if MODE=='TEST':
                        params['proportions'] = params['proportions'][len(idxs_train):,:]
                        params['theta_fd'] = params['theta_fd'][len(idxs_train):]
                        data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
                        postmeans = model.get_posterior_mean_latentvar(params, nsamples=800)
                        params.update(postmeans)
                        res = model.sampling(data_test, params)
                        data_y = res['frac_mean_r']
                        data_x = data_test['frac_r']
                        data_x = torch.sum(data_test['masks']['R']*torch.nan_to_num(data_x), dim=0) / torch.sum(data_test['masks']['R'], dim=0) 
                    else:
                        params['proportions'] = params['proportions'][:len(idxs_train),:]
                        params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                        _, data_train, _, _ = model.get_real_data_split(idxs_test, idxs_train)
                        postmeans = model.get_posterior_mean_latentvar(params, nsamples=800)
                        params.update(postmeans)
                        res = model.sampling(data_train, params)
                        data_y = res['frac_mean_r']
                        data_x = data_train['frac_r']
                        data_x = torch.sum(data_train['masks']['R']*torch.nan_to_num(data_x), dim=0) / torch.sum(data_train['masks']['R'], dim=0) 
                    data_x, data_y = np.array(data_x.reshape(-1)), np.array(data_y.reshape(-1))
                    idxs = np.where(~np.isnan(data_x))[0]

                    all_data[0][ite] = list(data_x[idxs])
                    all_data[1][ite] = list(data_y[idxs])
                    corrs[ite] = np.round(np.corrcoef(data_x[idxs], data_y[idxs])[0,1]**2, 6)
                count += 1
        except:
            print(ite)
            pass
    beta2_mean *= nbites / count
    beta_mean *= nbites / count
    beta_std = beta2_mean - beta_mean**2
    return all_data, corrs

In [ ]:
all_data_scatrex, dic_corrs_scatrex = get_stat(model_scatrex, COHORT, gene_set_collection, 'scatrex')
all_data_pheno, dic_corrs_pheno = get_stat(model_phenograph, COHORT, gene_set_collection, 'phenograph')

all_data_scatrex_train, dic_corrs_scatrex_train = get_stat(model_scatrex, COHORT, gene_set_collection, 'scatrex', MODE='TRAIN')
all_data_pheno_train, dic_corrs_pheno_train = get_stat(model_phenograph, COHORT, gene_set_collection, 'phenograph', MODE='TRAIN')

In [ ]:
corrs_scatrex_train, corrs_scatrex, corrs_pheno_train, corrs_pheno = [],[],[],[]
for ite in range(100):
    if (ite in dic_corrs_scatrex_train) and (ite in dic_corrs_pheno_train):
        corrs_scatrex_train.append(dic_corrs_scatrex_train[ite])
        corrs_pheno_train.append(dic_corrs_pheno_train[ite])
    if (ite in dic_corrs_scatrex) and (ite in dic_corrs_pheno):
        # if dic_corrs_gene[ite]>=0.5:
        corrs_scatrex.append(dic_corrs_scatrex[ite])
        corrs_pheno.append(dic_corrs_pheno[ite])

In [ ]:
np.array(corrs_scatrex)>=np.array(corrs_pheno)

In [ ]:
print('Scatrex is better than scRNA clusters {0}% of time'.format(100*np.mean(np.array(corrs_scatrex)>=np.array(corrs_pheno))))

In [ ]:
plt.scatter(all_data_pheno_train[0][0], all_data_pheno_train[1][0])

In [ ]:
a = corrs_scatrex_train
b = corrs_pheno_train
m = np.min([np.min(a), np.min(b)])
M = np.max([np.max(a), np.max(b)])
plt.ylim(m,M)
plt.xlim(m,M)
plt.scatter(a, b)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

for MODE in ['TRAIN','TEST']:
    # Data
    if MODE == 'TRAIN':
        a = corrs_pheno_train   # Correlation coefficients for SCATrEX
        b = corrs_scatrex_train # Correlation coefficients for Phenograph
        color = 'crimson'
        title = 'pheno_vs_scatrex_train.png'
    else:
        a = corrs_pheno   # Correlation coefficients for SCATrEX
        b = corrs_scatrex    # Correlation coefficients for Phenograph
        color = 'royalblue'
        title = 'pheno_vs_scatrex.png'
    m = np.min([np.min(a), np.min(b)])
    M = np.max([np.max(a), np.max(b)])

    # Wilcoxon signed-rank test
    stat, p_value = wilcoxon(a, b, alternative='two-sided')

    # Create jointplot

    g = sns.jointplot(x=a, y=b, kind='scatter', color=color, marginal_kws=dict(bins=20, fill=True))

    # Add y = x reference line
    ax = g.ax_joint
    ax.plot([m, M], [m, M], linestyle="dashed", color="black", linewidth=1.5, label="y = x (Reference)")

    # Add legend for y = x line
    ax.legend(loc="lower right", fontsize=12)

    # Add p-value as text annotation
    ax.text(0.47, 0.2, f'Wilcoxon p-value: {p_value:.3g}', transform=ax.transAxes, fontsize=12, 
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='black'))

    # Set axis labels
    g.set_axis_labels("scRNA clusters", "Integrated scDNA and scRNA tumour clones", fontsize=14)

    plt.savefig(title, dpi=250, bbox_inches='tight')
    # Show plot
    plt.show()

# Comparing gene only versus scvi

In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['gene','c6','hallmarks', 'c2_pid', 'c2_kegg_medicus']
clonemodes = ['scatrex', 'phenograph']
clonemode = 'scatrex'

if False:
    gene_set_collection = gene_set_collections[0]
    mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
    model_gene = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
    data_ref = model_gene.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


gene_set_collection_pathway = gene_set_collections[2]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection_pathway, clonemode)
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
model_pathway = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref_phenograph = model_pathway.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

gene_set_collection_scvi = "geneOncoKB"
mode_features = 'metacells_{0}_{1}_rawcounts_scvi'.format(gene_set_collection_scvi, clonemode)
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
model_scvi = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref_scvi = model_scvi.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

In [ ]:
add_clone = ''
#dic_gene, dic_corrs_gene = get_stat(model_gene, COHORT, gene_set_collection, clonemode+add_clone, 12.8, 3.2)
dic_scvi, dic_corrs_scvi = get_stat(model_scvi, COHORT, gene_set_collection_scvi, clonemode+'_rawcounts_scvi', 1.0, 1.0)

#dic_gene_train, dic_corrs_gene_train = get_stat(model_gene, COHORT, gene_set_collection, clonemode+add_clone, 12.8, 3.2, MODE='TRAIN')
dic_scvi_train, dic_corrs_scvi_train = get_stat(model_scvi, COHORT, gene_set_collection_scvi, clonemode+'_rawcounts_scvi', 1.0, 1.0, MODE='TRAIN')

dic_pathway, dic_corrs_pathway = get_stat(model_pathway, COHORT, gene_set_collection_pathway, clonemode, 3.2 , 3.2)
dic_pathway_train, dic_corrs_pathway_train = get_stat(model_pathway, COHORT, gene_set_collection_pathway, clonemode, 3.2, 3.2, MODE='TRAIN')

In [ ]:
corrs_scvi_train, corrs_scvi, corrs_pathway_train, corrs_pathway = [],[],[],[]
for ite in range(100):
    if (ite in dic_corrs_scvi_train) and (ite in dic_corrs_pathway_train):
        corrs_scvi_train.append(dic_corrs_scvi_train[ite])
        corrs_pathway_train.append(dic_corrs_pathway_train[ite])
        if (ite in dic_corrs_scvi) and (ite in dic_corrs_pathway):
                corrs_scvi.append(dic_corrs_scvi[ite])
                corrs_pathway.append(dic_corrs_pathway[ite])

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

for MODE in ['TRAIN','TEST']:
    # Data
    if MODE == 'TRAIN':
        a = corrs_pathway_train  # Correlation coefficients for SCATrEX
        b = corrs_scvi_train    # Correlation coefficients for Phenograph
        color = 'crimson'
        title = 'pathway_vs_scvi_train.png'
    else:
        a = corrs_pathway # Correlation coefficients for SCATrEX
        b = corrs_scvi    # Correlation coefficients for Phenograph
        color = 'royalblue'
        title = 'pathway_vs_scvi.png'
    m = np.min([np.min(a), np.min(b)])
    M = np.max([np.max(a), np.max(b)])

    # Wilcoxon signed-rank test
    stat, p_value = wilcoxon(a, b, alternative='two-sided')

    # Create jointplot

    g = sns.jointplot(x=a, y=b, kind='scatter', color=color, marginal_kws=dict(bins=20, fill=True))

    # Add y = x reference line
    ax = g.ax_joint
    ax.plot([m, M], [m, M], linestyle="dashed", color="black", linewidth=1.5, label="y = x (Reference)")

    # Add legend for y = x line
    ax.legend(loc="lower right", fontsize=12)

    # Add p-value as text annotation
    ax.text(0.47, 0.2, f'Wilcoxon p-value: {p_value:.3g}', transform=ax.transAxes, fontsize=12, 
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='black'))

    # Set axis labels
    g.set_axis_labels("Features at the pathway level", "Low-dimensional gene features", fontsize=14)

    plt.savefig(title, dpi=250, bbox_inches='tight')
    # Show plot
    plt.show()

# Comparing gene vs Pathways

In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['gene','c6','hallmarks', 'c2_pid', 'c2_kegg_medicus']
clonemodes = ['scatrex', 'phenograph']
gene_set_collection = gene_set_collections[0]
clonemode = 'scatrex'
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
    
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
concentration_DMSO = '100'
concentration_drug = '5'
path_info_cohort = None
model_gene = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref = model_gene.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

gene_set_collection_pathway = gene_set_collections[2]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection_pathway, clonemode)
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
model_pathway = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref_phenograph = model_pathway.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

In [ ]:
add_clone = ''
dic_gene, dic_corrs_gene = get_stat(model_gene, COHORT, gene_set_collection, clonemode+add_clone, 12.8, 3.2)
dic_pathway, dic_corrs_pathway = get_stat(model_pathway, COHORT, gene_set_collection_pathway, clonemode, 3.2 , 3.2)

dic_gene_train, dic_corrs_gene_train = get_stat(model_gene, COHORT, gene_set_collection, clonemode+add_clone, 12.8, 3.2, MODE='TRAIN')
dic_pathway_train, dic_corrs_pathway_train = get_stat(model_pathway, COHORT, gene_set_collection_pathway, clonemode, 3.2, 3.2, MODE='TRAIN')

In [ ]:
corrs_gene_train, corrs_gene, corrs_pathway_train, corrs_pathway = [],[],[],[]
for ite in range(100):
    if (ite in dic_corrs_gene_train) and (ite in dic_corrs_pathway_train):
        corrs_gene_train.append(dic_corrs_gene_train[ite])
        corrs_pathway_train.append(dic_corrs_pathway_train[ite])
        if (ite in dic_corrs_gene) and (ite in dic_corrs_pathway):
            if dic_corrs_gene[ite]>=0.5:
                corrs_gene.append(dic_corrs_gene[ite])
                corrs_pathway.append(dic_corrs_pathway[ite])

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

for MODE in ['TRAIN','TEST']:
    # Data
    if MODE == 'TRAIN':
        a = corrs_gene_train  # Correlation coefficients for SCATrEX
        b = corrs_pathway_train    # Correlation coefficients for Phenograph
        color = 'crimson'
        title = 'gene_vs_pathway_train.png'
    else:
        a = corrs_gene  # Correlation coefficients for SCATrEX
        b = corrs_pathway    # Correlation coefficients for Phenograph
        color = 'royalblue'
        title = 'gene_vs_pathway.png'
    m = np.min([np.min(a), np.min(b)])
    M = np.max([np.max(a), np.max(b)])

    # Wilcoxon signed-rank test
    stat, p_value = wilcoxon(a, b, alternative='two-sided')

    # Create jointplot

    g = sns.jointplot(x=a, y=b, kind='scatter', color=color, marginal_kws=dict(bins=20, fill=True))

    # Add y = x reference line
    ax = g.ax_joint
    ax.plot([m, M], [m, M], linestyle="dashed", color="black", linewidth=1.5, label="y = x (Reference)")

    # Add legend for y = x line
    ax.legend(loc="lower right", fontsize=12)

    # Add p-value as text annotation
    ax.text(0.47, 0.2, f'Wilcoxon p-value: {p_value:.3g}', transform=ax.transAxes, fontsize=12, 
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='black'))

    # Set axis labels
    g.set_axis_labels("Features at the gene level", "Features at the pathway level", fontsize=14)

    plt.savefig(title, dpi=250, bbox_inches='tight')
    # Show plot
    plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# Data (Assuming these variables are defined in your environment)
a_train = corrs_gene_train  
b_train = corrs_pathway_train
a_test = corrs_gene  
b_test = corrs_pathway

# Compute min/max for axis limits
m = np.min([np.min(a_train), np.min(b_train), np.min(a_test), np.min(b_test)])
M = np.max([np.max(a_train), np.max(b_train), np.max(a_test), np.max(b_test)])

# Compute Wilcoxon signed-rank test (one-sided: testing if Phenograph > SCATrEX)
stat_train, p_value_train = wilcoxon(a_train, b_train, alternative='greater')
stat_test, p_value_test = wilcoxon(a_test, b_test, alternative='greater')

# Create the joint plot
g = sns.jointplot(x=a_train, y=b_train, kind='scatter', color='crimson', marginal_kws=dict(bins=20, fill=True, alpha=0.5))
g.x = a_test  # Add test data
g.y = b_test
sns.scatterplot(x=a_test, y=b_test, ax=g.ax_joint, color='royalblue', alpha=0.6)  # Overlay test data

# Overlay histograms
sns.histplot(a_test, color='royalblue', alpha=0.4, bins=20, ax=g.ax_marg_x)
sns.histplot(b_test, color='royalblue', alpha=0.4, bins=20, ax=g.ax_marg_y)

sns.histplot(a_train, color='crimson', alpha=0.4, bins=20, ax=g.ax_marg_x)
sns.histplot(b_train, color='crimson', alpha=0.4, bins=20, ax=g.ax_marg_y)

# Add y = x reference line
ax = g.ax_joint
ax.plot([m, M], [m, M], linestyle="dashed", color="black", linewidth=1.5, label="y = x (Reference)")
ax.set_ylim([m, M])  # Set limits to match the scatter plot

# Set axis labels
g.set_axis_labels("Features at the gene level", "Features at the pathway level", fontsize=14)

# Save and show plot
plt.savefig("merged_gene_vs_pathway.png", dpi=250, bbox_inches='tight')
plt.show()


# UQ under parametric bootstrap

In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['gene','c6','hallmarks', 'c2_pid', 'c2_kegg_medicus']
clonemodes = ['scatrex', 'phenograph']
gene_set_collection = gene_set_collections[3]
clonemode = 'scatrex'
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
    
path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
concentration_DMSO = '100'
concentration_drug = '5'
path_info_cohort = None
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


In [ ]:

import pyro
def get_stat(clonemode_stat, model, MODE='TEST', drug='Elesclomol'):
    ite = 0
    found = False
    while (not(found) and ite<10):
        try:
            with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode_stat, ite)), 'rb') as handle:
                params = pickle.load(handle)
                dim = params['beta'].shape[1]
            found = True
        except:
            ite+=1
            pass

    param_studied = 'beta' # ['beta', 'pi']        

    if param_studied == 'beta':
        beta_mean = np.zeros((47,dim))
        beta2_mean = np.zeros((47,dim))
    else:
        D = params['beta'].shape[0]
        N, Kmax = params['proportions'].shape
        beta_mean = np.zeros((D, Kmax, 30))
        beta2_mean = np.zeros((D, Kmax, 30))
    
    count = 0
    all_data = [[],[]]
    corrs = []
    nbites= 80
    
    idxdrug = np.where(model.FD.selected_drugs==drug)[0][0]
    for itebis in range(nbites):
        #try:
            ite = 1
            with open(os.path.join(rootpath,'{0}/{1}_{2}/hyperparam_optim/params_svi_ite_{3}_l1_6.4_l2_3.2.pkl'.format(COHORT, gene_set_collection, clonemode_stat, ite)), 'rb') as handle:
                params = pickle.load(handle)
                if param_studied == 'beta':
                    beta_mean += params['beta'] /nbites
                    beta2_mean += params['beta']**2 /nbites
                    
                    samples_names_train = params['sample_names_train']
                    samples_names_test = params['sample_names_test']

                    idxs_train = []
                    idxs_test = []
                    for sample in samples_names_train:
                        idx_s = np.where(model.sample_names==sample)[0][0]
                        idxs_train.append(idx_s)
                    for sample in samples_names_test:
                        idx_s = np.where(model.sample_names==sample)[0][0]
                        idxs_test.append(idx_s)

                    pyro.set_rng_seed(itebis)

                    
                    if MODE=='TEST':
                        params['proportions'] = params['proportions'][len(idxs_train):,:]
                        params['theta_fd'] = params['theta_fd'][len(idxs_train):]
                        data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
                        postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                        params.update(postmeans)
                        res = model.sampling(data_test, params)
                        data_y = res['frac_mean_r']
                        data_x = data_test['frac_r']
                        data_x = torch.sum(data_test['masks']['R']*torch.nan_to_num(data_x), dim=0) / torch.sum(data_test['masks']['R'], dim=0) 
                    else:
                        params['proportions'] = params['proportions'][:len(idxs_train),:]
                        params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                        _, data_train, _, _ = model.get_real_data_split(idxs_test, idxs_train)
                        postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                        params.update(postmeans)
                        res = model.sampling(data_train, params)
                        data_y = res['frac_r'][0,:,:]
                        data_x = data_train['frac_r']
                        data_x = torch.sum(data_train['masks']['R']*torch.nan_to_num(data_x), dim=0) / torch.sum(data_train['masks']['R'], dim=0) 

                    data_x = data_x[idxdrug,:]
                    data_y = data_y[idxdrug,:]
                    data_x, data_y = np.array(data_x.reshape(-1)), np.array(data_y.reshape(-1))
                    idxs = np.where(~np.isnan(data_x))[0]

                    all_data[0].append(list(data_x[idxs]))
                    all_data[1].append(list(data_y[idxs]))
                    corrs.append(np.round(np.corrcoef(data_x[idxs], data_y[idxs])[0,1]**2, 6))
                count += 1
#         except:
#             pass
    beta2_mean *= nbites / count
    beta_mean *= nbites / count
    beta_std = beta2_mean - beta_mean**2
    return all_data, corrs

In [ ]:
drug = "Elesclomol"
all_data, corrs = get_stat('scatrex', model, drug=drug)
all_data_train, corrs_train = get_stat('scatrex', model, MODE='TRAIN', drug=drug)

In [ ]:
predall_data = np.array(all_data[1])
predall_data_train = np.array(all_data_train[1])
n_train = predall_data_train.shape[1]
preds = np.concatenate((predall_data_train, predall_data), axis=1)

obsall_data = np.array(all_data[0])
obsall_data_train = np.array(all_data_train[0])
obs = np.concatenate((obsall_data_train, obsall_data), axis=1)

vals, xs = [],[]
for i in range(preds.shape[1]):
    vals.append(preds[:,i])
    xs.append(obs[0,i])  # adds jitter to the data points - can be adjusted
    
plt.figure(figsize=(20,9))
for i in range(preds.shape[1]):
    xs = [obs[0,i] for a in range(preds.shape[0])]
    if i<obsall_data_train.shape[1]:
        c = 'red'
    else:
        c = 'blue'
    plt.scatter(xs, preds[:,i],  alpha=0.4, color=c)
plt.plot([0,1],[0,1], linestyle='--', color='black')

In [ ]:
# Study calibration
import bisect
from tqdm import tqdm

training = []
testing = []
for idxdrug, drug in tqdm(enumerate(model.FD.selected_drugs)):
    all_data, corrs = get_stat('scatrex', model, drug=drug)
    all_data_train, corrs_train = get_stat('scatrex', model, MODE='TRAIN', drug=drug)
    predall_data = np.array(all_data[1])
    predall_data_train = np.array(all_data_train[1])
    obsall_data = np.array(all_data[0])
    obsall_data_train = np.array(all_data_train[0])
    
    for itrain in range(predall_data_train.shape[1]):
        sorted_list = np.sort(predall_data_train[:,itrain])
        insertion_index = bisect.bisect_left(sorted_list, obsall_data_train[0,itrain])
        training.append(insertion_index/len(sorted_list))
    for itest in range(predall_data.shape[1]):
        sorted_list = np.sort(predall_data[:,itest])
        insertion_index = bisect.bisect_left(sorted_list, obsall_data[0,itest])
        testing.append(insertion_index/len(sorted_list))

In [ ]:
alphas = np.arange(0.01,0.3, 0.03)
props_training = []
props_testing = []
for alpha in alphas:
    props_training.append(np.mean((training>alpha/2) & (training<(1-alpha/2))))
    props_testing.append(np.mean((testing>alpha/2) & (testing<(1-alpha/2))))

In [ ]:
y=props_testing
m = np.min(1-alphas)
m = np.min([m, np.min(y)])
M = np.max(1-alphas)
M = np.max([M, np.max(y)])
plt.xlim(m,M)
plt.ylim(m,M)
plt.plot([m,M],[m,M], color='black', linestyle='--', alpha=0.5)
plt.scatter(1-alphas, y, color='red', marker='x')
plt.xlabel('Targeted marginal coverage', fontsize=14)
plt.ylabel('Empirical marginal coverage', fontsize=14)
plt.savefig('coverage.png', dpi=250, bbox_inches='tight')

In [ ]:
for drug in model.FD.selected_drugs:
    all_data, corrs = get_stat('scatrex', model, drug=drug)
    all_data_train, corrs_train = get_stat('scatrex', model, MODE='TRAIN', drug=drug)
    predall_data = np.array(all_data[1])
    predall_data_train = np.array(all_data_train[1])
    n_train = predall_data_train.shape[1]
    preds = np.concatenate((predall_data_train, predall_data), axis=1)

    obsall_data = np.array(all_data[0])
    obsall_data_train = np.array(all_data_train[0])
    obs = np.concatenate((obsall_data_train, obsall_data), axis=1)


    import matplotlib.pyplot as plt
    import numpy as np

    plt.figure(figsize=(11, 6))

    # Extract observed values (X-axis) and corresponding prediction distributions (Y-axis)
    xs = np.array([obs[0, i] for i in range(preds.shape[1])])
    data = [preds[:, i] for i in range(preds.shape[1])]

    # Sort by observed values (xs)
    colors = ['red' for i in range(obsall_data_train.shape[1])] + ['blue' for i in range(obsall_data.shape[1])]
    sorted_indices = np.argsort(xs)
    xs_sorted = xs[sorted_indices]
    data_sorted = [data[i] for i in sorted_indices]
    colors = np.array(colors)[sorted_indices]

    # Scale positions for correct placement on the x-axis
    xs_positions = np.array([100 * el for el in xs_sorted])

    # Use Matplotlib's boxplot with manual positions
    plt.boxplot(data_sorted, positions=xs_positions, widths=0.5, whis=[5, 95], showfliers=False)

    # Adjust x-axis ticks and labels
    tics = np.arange(0.1,0.9,0.1)
    plt.xticks(ticks=tics*100, labels=np.round(tics, 2), fontsize=14,)
    plt.xlabel("Observed fraction of tumor cells", fontsize=14)
    plt.ylabel("Predicted fraction of tumor cells", fontsize=14)
    #plt.title("Prediction Uncertainty with Boxplots (Ordered by Observed Values)")

    # Ensure same limits for x and y
    min_val, max_val = min(xs_positions), max(xs_positions)
    plt.xlim(min_val, max_val)  # Add padding
    plt.ylim(min_val / 100, max_val / 100)

    # Reference y=x line
    plt.plot([min_val, max_val], [min_val / 100, max_val / 100], linestyle='--', color='black', label="y=x Reference Line")


    for i in range(preds.shape[1]):
        plt.scatter(xs_positions[i]*np.ones(preds.shape[0]), data_sorted[i],  alpha=0.1, s=10, color=colors[i])

    plt.scatter([], [],  alpha=0.9, s=40, color='red', label='Training samples')
    plt.scatter([], [],  alpha=0.9, s=40, color='blue', label='Test samples')
    plt.yticks(fontsize=16)


    plt.legend(fontsize=14)
    plt.savefig('UQ_parametricBS_{0}.png'.format(drug), dpi=250, bbox_inches='tight')

    plt.show()


In [ ]:
rootpath = '/data/users/quentin/final_package/experiments/paper_results'
penalties_l1 = [0.2*2**i for i in range(10)]
penalties_l2 = [0.05*2**i for i in range(10)]
log_likeli = np.zeros((len(penalties_l1),len(penalties_l2)))
variances = np.zeros((len(penalties_l1),len(penalties_l2)))

for pl1, penalty_l1 in tqdm(enumerate(penalties_l1)):
    for pl2, penalty_l2 in enumerate(penalties_l2):
        beta = np.zeros((47,500))
        count = 0
        log_likeli_temp = 0
        for ite in range(4):
            try:
                with open(os.path.join(rootpath,'{0}/{1}_{2}/stability/params_svi_ite_{5}_l1_{3}_l2_{4}.pkl'.format(COHORT, gene_set_collection, clonemode, str(penalty_l1), str(penalty_l2), ite)), 'rb') as handle:
                    params = pickle.load(handle)
                    beta += (params['beta']**2-params['beta'])
                    count += 1
                samples_names_train = params['sample_names_train']
                idxs_train = []
                idxs_test = []
                for sample in samples_names_train:
                    idx_s = np.where(model.sample_names==sample)[0][0]
                    idxs_train.append(idx_s)
                params['proportions'] = params['proportions'][:len(idxs_train),:]
                params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, [])
                postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                params.update(postmeans)
                log_likeli_temp += model.get_log_likelihood(data_train, {key:val for key,val in params.items() if not('sample_' in key)})
            except:
                print('error', pl1, pl2)
                pass
        beta /= count
        variances[pl1, pl2] = np.mean(np.sort(beta.reshape(-1))[-100:])#np.mean(beta)
        log_likeli[pl1, pl2] = log_likeli_temp/count

In [ ]:
penalties_l1 = [0.2*2**i for i in range(10)]
penalties_l2 = [0.05*2**i for i in range(10)]
print(penalties_l1, penalties_l2)

# Visualization drug independent culture effect

In [ ]:
from scipy.interpolate import BSpline
import skfda

with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    params = pickle.load(handle)
                
data_train, data_test, sample_names_train, sample_names_test = model_scatrex.get_real_data_split(range(data_ref['N']), [])

def get_spline_basis(x, degree=3, nknots=10):
    k = nknots
    knots = np.linspace(0, 1, k - degree)
    bsplines = skfda.representation.basis.BSplineBasis(knots=knots, order=degree + 1)
    res = np.zeros((len(x),k-1))
    for i in range(k-1):
        res[:,i] = bsplines.to_basis().tolist()[i](x).reshape(-1)
    return res


MAX_nc = data_train['n_c'].reshape(-1).max()

features = ['frac_rna', 'density', 'intercept_pipet', 'intercept_pipet', 'intercept'] 
degree = 3
nknots = 5
feat_dim = nknots-1
feat_dim_global = 1 + (len(features)-1)*feat_dim

for idf, feature in enumerate(features[:-1]):
    idxs = [l for l in range(feat_dim*idf,feat_dim*(idf+1))]
    x_nu = np.linspace(0,1,100)
    X_nu = list(get_spline_basis(x_nu, degree=degree, nknots=nknots))
    nu_tumor_over_nu_healthy = np.exp(X_nu @ params['beta_control'][idxs])
    nu_healthy_c = 1/(1+nu_tumor_over_nu_healthy)
    plt.plot(x_nu, nu_healthy_c)
    plt.title(feature)
    plt.show()

# X = np.zeros((len(sample_names_train)+len(sample_names_test), data_train['C'], feat_dim_global))
# Xdrug = np.zeros((len(sample_names), data_train['D'], data_train['R'], feat_dim_global))


# for idsample, sample in enumerate(sample_names):
#     feat_vec = []
#     feat_vec.append(frac_rna[idsample])
#     wellpos = self.FD.sample2control_wellpos[sample]
#     for i in range(wellpos.shape[0]):
#         well1 = wellpos[i,0]-1
#         well2 = wellpos[i,1]-1
#         feat_vec2 = list(feat_vec)
#         feat_vec2 += [data_train['n_c'][i, idsample] / MAX_nc]
#         feat_vec2 += [well2%2 * well2/24, (well2+1)%2 * well2/24]
#         feat_vec2 = list(get_spline_basis(feat_vec2, degree=degree, nknots=nknots))
#         feat_vec2 += [1]
#         X[idsample,i,:] = feat_vec2

#     for d,drug in enumerate(self.FD.selected_drugs):
#         wellpos = self.FD.sample2wellpos[(drug,sample)]
#         for i in range(wellpos.shape[0]):
#             well1 = wellpos[i,0]-1
#             well2 = wellpos[i,1]-1
#             feat_vec2 = list(feat_vec)
#             feat_vec2 += [data_train['n_c'][i, idsample] / MAX_nc]
#             feat_vec2 += [well2%2 * well2/24, (well2+1)%2 * well2/24]
#             feat_vec2 = list(get_spline_basis(feat_vec2, degree=degree, nknots=nknots))
#             feat_vec2 += [ 1]
#             Xdrug[idsample,d,i,:] = feat_vec2

# for idsample, sample in enumerate(sample_names):
#     feat_vec = []
#     feat_vec.append(frac_rna[idsample])
#     wellpos = model_scatrex.FD.sample2control_wellpos[sample]
#     for i in range(wellpos.shape[0]):
#         well1 = wellpos[i,0]-1
#         well2 = wellpos[i,1]-1
#         feat_vec2 = list(feat_vec)
#         feat_vec2 += [data_train['n_c'][i, idsample] / MAX_nc]
#         feat_vec2 += [well2%2 * well2/24, (well2+1)%2 * well2/24]
#         feat_vec2 = list(get_spline_basis(feat_vec2, degree=degree, nknots=nknots))
#         feat_vec2 += [1]
#         X[idsample,i,:] = feat_vec2